# Projeto de Integração — VRA/ANAC 2020
## Notebook 01 — Exploração e diagnóstico dos dados (sem limpeza)

**Objetivo:** ler os 12 CSVs locais de `dados/brutos/2020/`, preservar os valores e registrar evidências para o relatório, seguindo a sequência do notebook `exploracao-2019.ipynb`.

Nenhuma linha é removida, rejeitada ou carregada em banco. Conversões temporárias servem apenas para validar formatos. Este notebook não implementa o pipeline ETL.

**Contexto do repositório:** o README descreve uma estrutura planejada com exploração, ETL, ELT, SQL e documentação. No checkout analisado existem os CSVs de 2020, a exploração e o ELT de 2019 e rascunhos de 2020. O rascunho `exploracao-2020.ipynb` é preservado; este arquivo usa o nome solicitado, com underscore.

**Limite documental:** `documentacao/contrato_dados.md`, `dicionario_dados.md` e `esquema_estrela.drawio` são citados no README, mas não estão disponíveis. Não foi possível conferir a modelagem dimensional nem um contrato independente. As regras abaixo são **provisórias**, extraídas dos notebooks de 2019; não representam aprovação do modelo conjunto.

**Referência compartilhada pelo grupo:** [ANAC_2019_01_Exploracao.ipynb](https://colab.research.google.com/drive/1wZ9010xnFKPCOSu7y1_Mo_5AbETkcIIY?usp=sharing), conferida e equivalente em conteúdo ao notebook de exploração local de 2019. O link contém um notebook de exploração, sem esquema dimensional ou contrato independente.

**Escopo confirmado pelo integrante:** exploração de 2020 para preparar seu futuro ETL. A divisão de trabalho enviada menciona originalmente 2023–2025, enquanto a estrutura local e a confirmação do integrante adotam 2019–2021; este notebook mantém 2020, sem redefinir responsabilidades da equipe. Segundo relato do integrante, o grupo recebeu orientação para excluir repetições exatas no tratamento. Aqui apenas identificamos essas ocorrências; a remoção ficará restrita à futura camada tratada, preservando os CSVs originais.


### 1. Preparar o ambiente
Usamos Pandas e a biblioteca padrão, sem instalar pacotes ou exigir Colab. `dtype=str`, `keep_default_na=False` e `na_filter=False` preservam zeros à esquerda, vazios e marcadores textuais como `NA`. `skip_blank_lines=False` evita descartar silenciosamente registros vazios. Espaços, caixa, acentos e datas permanecem como recebidos.

In [1]:
from pathlib import Path
from collections import Counter
import csv
import hashlib
import re
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 25)
pd.set_option("display.max_rows", 30)
print("Pandas:", pd.__version__)

Pandas: 3.1.1


### 2. Localizar os arquivos originais
Execute a partir da raiz do projeto ou de `notebooks/exploracao/`. A raiz é procurada nos diretórios ancestrais. Exigimos exatamente os 12 nomes mensais, sem aceitar meses repetidos, ausentes ou arquivos inesperados. SHA-256 documenta os bytes de cada CSV e será conferido novamente ao final.

In [2]:
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dados/brutos/2020").is_dir()), None)
if raiz is None:
    raise FileNotFoundError("Execute dentro do repositório com dados/brutos/2020/ disponível.")
data_dir = raiz / "dados/brutos/2020"
arquivos = sorted(p for p in data_dir.iterdir() if p.suffix.lower() == ".csv")
nomes_esperados = {f"VRA_2020_{mes:02d}.csv" for mes in range(1, 13)}
assert {p.name for p in arquivos} == nomes_esperados, "Confira os 12 nomes mensais de 2020."
def sha256_arquivo(caminho):
    with caminho.open("rb") as fonte:
        return hashlib.file_digest(fonte, "sha256").hexdigest()
hashes_iniciais = {p.name: sha256_arquivo(p) for p in arquivos}
print("Arquivos encontrados:", len(arquivos))

Arquivos encontrados: 12


### 3. Identificar e ler os arquivos sem alterar dados
Conferimos cabeçalhos e largura de **todos** os registros com `csv.reader` antes de concatenar: linhas curtas não devem virar ausências artificiais e cabeçalhos repetidos não devem ser renomeados silenciosamente pelo Pandas. A decodificação UTF-8 é estrita; `utf-8-sig` aceita BOM opcional. Não há fallback silencioso de encoding.

O cabeçalho esperado vem da lista explícita no ELT de 2019. Diferenças mensais são exibidas antes de bloquear a análise que depende desse esquema. Acrescentamos somente `arquivo_origem` e `linha_origem`: a segunda é a posição do **registro lógico** mais o cabeçalho, não necessariamente a linha física quando há quebras dentro de campos citados.

In [3]:
colunas_esperadas = [
    "Sigla ICAO Empresa Aérea", "Empresa Aérea", "Número Voo", "Código DI",
    "Código Tipo Linha", "Modelo Equipamento", "Número de Assentos",
    "Sigla ICAO Aeroporto Origem", "Descrição Aeroporto Origem", "Partida Prevista",
    "Partida Real", "Sigla ICAO Aeroporto Destino", "Descrição Aeroporto Destino",
    "Chegada Prevista", "Chegada Real", "Situação Voo", "Justificativa",
    "Referência", "Situação Partida", "Situação Chegada"
]
estrutura = []
for arquivo in arquivos:
    with arquivo.open(encoding="utf-8-sig", newline="") as fonte:
        leitor = csv.reader(fonte, delimiter=";", strict=True)
        cabecalho = next(leitor)
        larguras = Counter(len(registro) for registro in leitor)
    with arquivo.open("rb") as fonte:
        bom = fonte.read(3) == b"\xef\xbb\xbf"
    estrutura.append({
        "arquivo": arquivo.name, "registros": sum(larguras.values()),
        "colunas": len(cabecalho), "cabecalho_igual_2019": cabecalho == colunas_esperadas,
        "colunas_ausentes": sorted(set(colunas_esperadas) - set(cabecalho)),
        "colunas_extras": sorted(set(cabecalho) - set(colunas_esperadas)),
        "cabecalho_repetido": len(cabecalho) != len(set(cabecalho)),
        "registros_largura_divergente": sum(n for largura, n in larguras.items() if largura != len(cabecalho)),
        "utf8_bom": bom, "bytes": arquivo.stat().st_size,
        "sha256": hashes_iniciais[arquivo.name]
    })
resumo_estrutura = pd.DataFrame(estrutura)
display(resumo_estrutura)
assert resumo_estrutura["cabecalho_igual_2019"].all(), "Esquema divergente: revise o inventário acima."
assert resumo_estrutura["registros_largura_divergente"].eq(0).all(), "Há registros com largura inesperada."

partes = []
for arquivo in arquivos:
    mensal = pd.read_csv(arquivo, sep=";", encoding="utf-8-sig", dtype=str,
                         keep_default_na=False, na_filter=False, skip_blank_lines=False)
    assert mensal.columns.tolist() == colunas_esperadas
    assert len(mensal) == resumo_estrutura.set_index("arquivo").loc[arquivo.name, "registros"]
    mensal["arquivo_origem"] = arquivo.name
    mensal["linha_origem"] = range(2, len(mensal) + 2)
    partes.append(mensal)
df_2020 = pd.concat(partes, ignore_index=True)
colunas_originais = colunas_esperadas.copy()
del partes, mensal

def assinatura_dataframe(df):
    return hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()
assinatura_inicial = assinatura_dataframe(df_2020)
print("Registros:", len(df_2020), "| Colunas originais:", len(colunas_originais),
      "| Com rastreabilidade:", len(df_2020.columns))
display(df_2020.head(3))
display(pd.DataFrame({"coluna": colunas_originais,
                      "dtype_leitura": [str(df_2020[c].dtype) for c in colunas_originais]}))

,arquivo,registros,colunas,cabecalho_igual_2019,colunas_ausentes,colunas_extras,cabecalho_repetido,registros_largura_divergente,utf8_bom,bytes,sha256
0,VRA_2020_01.csv,92390,20,True,[],[],False,0,False,27455960,c64acd1a13397c83861acf5693bf60aa5550acf1fbad08...
1,VRA_2020_02.csv,80919,20,True,[],[],False,0,False,24326067,ba6a91885d1f1ded54cb46bb714d5a91b26294088a4f82...
2,VRA_2020_03.csv,64523,20,True,[],[],False,0,False,19232759,e6fe0059e5d784bf7e551bf14aa3e551771c4450dd047c...
3,VRA_2020_04.csv,7617,20,True,[],[],False,0,False,2113562,c76f5ea70b68896f06335eb54903e2b75cc57ee9b41b6e...
4,VRA_2020_05.csv,9514,20,True,[],[],False,0,False,2643533,20dfa011cb3423f92674baa629429dfb88b9950fa0a7c4...
5,VRA_2020_06.csv,12800,20,True,[],[],False,0,False,3578139,d01012074bcb98932f85e0c8ef76e02a2b1c0fc4a52200...
6,VRA_2020_07.csv,19850,20,True,[],[],False,0,False,5632871,883b2fc9da868733968ae13c03eb6da42834797b8acbfa...
7,VRA_2020_08.csv,24806,20,True,[],[],False,0,False,7057047,2c187990075bc37701abc3e1a0c4c0f6951cd928b1d34d...
8,VRA_2020_09.csv,30660,20,True,[],[],False,0,False,8737728,f23a155467ca9424ce6f74cf0acf4182650293aff0fabd...
9,VRA_2020_10.csv,40590,20,True,[],[],False,0,False,11539155,d12483fe7b950ace85c012c68ded986a9b8281018eaea2...


Registros: 483545 | Colunas originais: 20 | Com rastreabilidade: 22


,Sigla ICAO Empresa Aérea,Empresa Aérea,Número Voo,Código DI,Código Tipo Linha,Modelo Equipamento,Número de Assentos,Sigla ICAO Aeroporto Origem,Descrição Aeroporto Origem,Partida Prevista,Partida Real,Sigla ICAO Aeroporto Destino,Descrição Aeroporto Destino,Chegada Prevista,Chegada Real,Situação Voo,Justificativa,Referência,Situação Partida,Situação Chegada,arquivo_origem,linha_origem
0,AAL,"AMERICAN AIRLINES, INC.",0213,1,I,B752,173,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",01/01/2020 18:47,01/01/2020 18:47,SBBR,PRESIDENTE JUSCELINO KUBITSCHEK - BRASÍLIA - D...,02/01/2020 02:27,02/01/2020 02:27,REALIZADO,,2020-01-01,Pontual,Pontual,VRA_2020_01.csv,2
1,AAL,"AMERICAN AIRLINES, INC.",0216,0,I,B789,285,SBGR,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - ...,01/01/2020 22:30,01/01/2020 22:30,KLAX,LOS ANGELES INTERNATIONAL AIRPORT - LOS ANGELE...,02/01/2020 11:00,02/01/2020 11:00,REALIZADO,,2020-01-01,Pontual,Pontual,VRA_2020_01.csv,3
2,AAL,"AMERICAN AIRLINES, INC.",0904,0,I,B772,273,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,01/01/2020 22:25,01/01/2020 22:25,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",02/01/2020 07:00,02/01/2020 07:00,REALIZADO,,2020-01-01,Pontual,Pontual,VRA_2020_01.csv,4


,coluna,dtype_leitura
0,Sigla ICAO Empresa Aérea,str
1,Empresa Aérea,str
2,Número Voo,str
3,Código DI,str
4,Código Tipo Linha,str
5,Modelo Equipamento,str
6,Número de Assentos,str
7,Sigla ICAO Aeroporto Origem,str
8,Descrição Aeroporto Origem,str
9,Partida Prevista,str


### 4. Conferir valores ausentes
Distinguimos strings vazias, strings compostas somente de espaços e nulos técnicos (`isna`). O diagnóstico de ausência usa `strip()` numa expressão temporária; os textos originais não são sobrescritos. Marcadores como `NA`, `NULL`, `N/A` e `NaN` são contados separadamente como **candidatos**, sem assumir que o contrato os define como ausência.

In [4]:
nulos = df_2020[colunas_originais].isna()
vazios_literais = df_2020[colunas_originais].eq("")
ausencias = pd.DataFrame({c: df_2020[c].isna() | df_2020[c].str.strip().eq("")
                         for c in colunas_originais})
marcadores = {"NA", "N/A", "NULL", "NAN", "NONE"}
resumo_vazios = pd.DataFrame({
    "coluna": colunas_originais,
    "nulos_tecnicos": nulos.sum().values,
    "strings_vazias": vazios_literais.sum().values,
    "somente_espacos": (ausencias & ~nulos & ~vazios_literais).sum().values,
    "total_ausentes": ausencias.sum().values,
    "percentual_ausente": (ausencias.mean() * 100).round(4).values,
    "marcadores_textuais_candidatos": [int(df_2020[c].str.strip().str.upper().isin(marcadores).sum())
                                       for c in colunas_originais]
}).sort_values("total_ausentes", ascending=False)
display(resumo_vazios)
vazios_mensais = ausencias.groupby(df_2020["arquivo_origem"]).sum()
display(vazios_mensais)

,coluna,nulos_tecnicos,strings_vazias,somente_espacos,total_ausentes,percentual_ausente,marcadores_textuais_candidatos
16,Justificativa,0,417447,0,417447,86.3305,0
19,Situação Chegada,0,23375,0,23375,4.8341,0
18,Situação Partida,0,23375,0,23375,4.8341,0
14,Chegada Real,0,21868,0,21868,4.5224,0
10,Partida Real,0,21868,0,21868,4.5224,0
9,Partida Prevista,0,1512,0,1512,0.3127,0
13,Chegada Prevista,0,1512,0,1512,0.3127,0
3,Código DI,0,0,24,24,0.0050,0
5,Modelo Equipamento,0,5,0,5,0.0010,0
6,Número de Assentos,0,5,0,5,0.0010,0


,Sigla ICAO Empresa Aérea,Empresa Aérea,Número Voo,Código DI,Código Tipo Linha,Modelo Equipamento,Número de Assentos,Sigla ICAO Aeroporto Origem,Descrição Aeroporto Origem,Partida Prevista,Partida Real,Sigla ICAO Aeroporto Destino,Descrição Aeroporto Destino,Chegada Prevista,Chegada Real,Situação Voo,Justificativa,Referência,Situação Partida,Situação Chegada
arquivo_origem,,,,,,,,,,,,,,,,,,,,
VRA_2020_01.csv,0,0,0,0,0,0,0,0,0,0,1308,0,0,0,1308,0,70356,0,1308,1308
VRA_2020_02.csv,0,0,0,5,0,0,0,0,0,0,1805,0,0,0,1805,0,55936,0,1805,1805
VRA_2020_03.csv,0,0,0,19,0,5,5,0,0,5,4228,0,0,5,4228,0,45442,0,4228,4228
VRA_2020_04.csv,0,0,0,0,0,0,0,0,0,66,1390,0,0,66,1390,0,7617,0,1456,1456
VRA_2020_05.csv,0,0,0,0,0,0,0,0,0,70,1341,0,0,70,1341,0,9514,0,1411,1411
VRA_2020_06.csv,0,0,0,0,0,0,0,0,0,164,1304,0,0,164,1304,0,12800,0,1468,1468
VRA_2020_07.csv,0,0,0,0,0,0,0,0,0,253,1441,0,0,253,1441,0,19850,0,1694,1694
VRA_2020_08.csv,0,0,0,0,0,0,0,0,0,259,1453,0,0,259,1453,0,24806,0,1712,1712
VRA_2020_09.csv,0,0,0,0,0,0,0,0,0,242,1226,0,0,242,1226,0,30660,0,1468,1468


### 5. Diagnosticar duplicatas exatas
Comparamos as **20 colunas originais** em toda a base anual, excluindo os metadados. Não aplicamos `strip`, conversão de datas ou mudança de caixa antes da comparação. Diferenciamos grupos repetidos, todos os registros envolvidos e ocorrências excedentes. A contagem anual detecta repetições entre arquivos; a contagem local por mês permite distingui-las. Nenhuma linha é eliminada.

In [5]:
excedente = df_2020.duplicated(subset=colunas_originais, keep="first")
envolvida = df_2020.duplicated(subset=colunas_originais, keep=False)
duplicatas_excedentes = int(excedente.sum())
registros_envolvidos = int(envolvida.sum())
grupos_repetidos = registros_envolvidos - duplicatas_excedentes
print("Grupos repetidos:", grupos_repetidos)
print("Registros envolvidos:", registros_envolvidos)
print("Duplicatas excedentes:", duplicatas_excedentes)
print("Total hipotético após deduplicação:", len(df_2020) - duplicatas_excedentes)
por_mes = []
for nome, grupo in df_2020.groupby("arquivo_origem", sort=True):
    por_mes.append({"arquivo": nome, "registros": len(grupo),
                    "excedentes_no_arquivo": int(grupo.duplicated(subset=colunas_originais).sum()),
                    "excedentes_na_base_anual": int(excedente.loc[grupo.index].sum()),
                    "envolvidos_na_base_anual": int(envolvida.loc[grupo.index].sum())})
resumo_duplicatas = pd.DataFrame(por_mes)
display(resumo_duplicatas)
print("Grupos que aparecem em mais de um arquivo:",
      len(df_2020.loc[envolvida].groupby(colunas_originais, dropna=False)["arquivo_origem"]
          .nunique().loc[lambda s: s > 1]))
display(df_2020.loc[envolvida].head(10))

Grupos repetidos: 124
Registros envolvidos: 248
Duplicatas excedentes: 124
Total hipotético após deduplicação: 483421


,arquivo,registros,excedentes_no_arquivo,excedentes_na_base_anual,envolvidos_na_base_anual
0,VRA_2020_01.csv,92390,4,4,8
1,VRA_2020_02.csv,80919,30,30,60
2,VRA_2020_03.csv,64523,90,90,180
3,VRA_2020_04.csv,7617,0,0,0
4,VRA_2020_05.csv,9514,0,0,0
5,VRA_2020_06.csv,12800,0,0,0
6,VRA_2020_07.csv,19850,0,0,0
7,VRA_2020_08.csv,24806,0,0,0
8,VRA_2020_09.csv,30660,0,0,0
9,VRA_2020_10.csv,40590,0,0,0


Grupos que aparecem em mais de um arquivo: 0


,Sigla ICAO Empresa Aérea,Empresa Aérea,Número Voo,Código DI,Código Tipo Linha,Modelo Equipamento,Número de Assentos,Sigla ICAO Aeroporto Origem,Descrição Aeroporto Origem,Partida Prevista,Partida Real,Sigla ICAO Aeroporto Destino,Descrição Aeroporto Destino,Chegada Prevista,Chegada Real,Situação Voo,Justificativa,Referência,Situação Partida,Situação Chegada,arquivo_origem,linha_origem
11519,AZU,AZUL LINHAS AÉREAS BRASILEIRAS S/A,5213,0,N,E195,118,SBSL,MARECHAL CUNHA MACHADO - SÃO LUÍS - MA - BRASIL,10/01/2020 22:00,10/01/2020 22:25,SBBE,INTERNACIONAL DE BELÉM/VAL DE CANS/JÚLIO CEZAR...,10/01/2020 23:05,10/01/2020 23:31,REALIZADO,AEROPORTO ORIGEM ABAIXO DOS LIMITES,2020-01-10,Pontual,Pontual,VRA_2020_01.csv,11521
11520,AZU,AZUL LINHAS AÉREAS BRASILEIRAS S/A,5213,0,N,E195,118,SBSL,MARECHAL CUNHA MACHADO - SÃO LUÍS - MA - BRASIL,10/01/2020 22:00,10/01/2020 22:25,SBBE,INTERNACIONAL DE BELÉM/VAL DE CANS/JÚLIO CEZAR...,10/01/2020 23:05,10/01/2020 23:31,REALIZADO,AEROPORTO ORIGEM ABAIXO DOS LIMITES,2020-01-10,Pontual,Pontual,VRA_2020_01.csv,11522
90939,TCV,EMPRESA DE TRANSPORTES AÉREOS DE CABO VERDE - ...,0661,1,I,B752,183,GVAC,AEROPORTO INTERNACIONAL AMÍLCAR CABRAL - ILHA ...,22/01/2020 21:17,22/01/2020 21:17,SBSV,DEPUTADO LUÍS EDUARDO MAGALHÃES - SALVADOR - B...,23/01/2020 01:57,23/01/2020 01:57,REALIZADO,"ATRASOS NÃO ESPECÍFICOS, OUTROS",2020-01-22,Pontual,Pontual,VRA_2020_01.csv,90941
90940,TCV,EMPRESA DE TRANSPORTES AÉREOS DE CABO VERDE - ...,0661,1,I,B752,183,GVAC,AEROPORTO INTERNACIONAL AMÍLCAR CABRAL - ILHA ...,22/01/2020 21:17,22/01/2020 21:17,SBSV,DEPUTADO LUÍS EDUARDO MAGALHÃES - SALVADOR - B...,23/01/2020 01:57,23/01/2020 01:57,REALIZADO,"ATRASOS NÃO ESPECÍFICOS, OUTROS",2020-01-22,Pontual,Pontual,VRA_2020_01.csv,90942
90941,TCV,EMPRESA DE TRANSPORTES AÉREOS DE CABO VERDE - ...,0662,0,I,B752,183,SBSV,DEPUTADO LUÍS EDUARDO MAGALHÃES - SALVADOR - B...,22/01/2020 23:30,,GVAC,AEROPORTO INTERNACIONAL AMÍLCAR CABRAL - ILHA ...,23/01/2020 04:45,,CANCELADO,CANCELAMENTO POR MOTIVOS TÉCNICOS - OPERACIONAIS,2020-01-22,,,VRA_2020_01.csv,90943
90942,TCV,EMPRESA DE TRANSPORTES AÉREOS DE CABO VERDE - ...,0662,0,I,B752,183,SBSV,DEPUTADO LUÍS EDUARDO MAGALHÃES - SALVADOR - B...,22/01/2020 23:30,,GVAC,AEROPORTO INTERNACIONAL AMÍLCAR CABRAL - ILHA ...,23/01/2020 04:45,,CANCELADO,CANCELAMENTO POR MOTIVOS TÉCNICOS - OPERACIONAIS,2020-01-22,,,VRA_2020_01.csv,90944
92355,UPS,UPS - UNITED PARCEL SERVICE CO.,0410,0,G,B763,0,SCEL,COMODORO ARTURO MERINO BENÍTEZ INTERNATIONAL A...,23/01/2020 03:33,23/01/2020 02:57,SAEZ,MINISTRO PISTARINI INTERNATIONAL AIRPORT (EZEI...,23/01/2020 05:20,23/01/2020 04:38,REALIZADO,ANTECIPAÇÃO DE HORÁRIO AUTORIZADA - ESPECÍFICO...,2020-01-22,Antecipado,Antecipado,VRA_2020_01.csv,92357
92356,UPS,UPS - UNITED PARCEL SERVICE CO.,0410,0,G,B763,0,SCEL,COMODORO ARTURO MERINO BENÍTEZ INTERNATIONAL A...,23/01/2020 03:33,23/01/2020 02:57,SAEZ,MINISTRO PISTARINI INTERNATIONAL AIRPORT (EZEI...,23/01/2020 05:20,23/01/2020 04:38,REALIZADO,ANTECIPAÇÃO DE HORÁRIO AUTORIZADA - ESPECÍFICO...,2020-01-22,Antecipado,Antecipado,VRA_2020_01.csv,92358
92948,AEA,AIR EUROPA LINEAS AEREAS SOCIEDAD ANONIMA,0047,0,I,A332,299,LEMD,ADOLFO SUÁREZ MADRID-BARAJAS AIRPORT - MADRID ...,01/02/2020 11:40,01/02/2020 12:21,SBRF,GUARARAPES - GILBERTO FREYRE - RECIFE - PE - B...,01/02/2020 20:15,01/02/2020 20:00,REALIZADO,SEGURANÇA/PAX/CARGA/ALARME,2020-02-01,Atraso 30-60,Antecipado,VRA_2020_02.csv,560
92949,AEA,AIR EUROPA LINEAS AEREAS SOCIEDAD ANONIMA,0047,0,I,A332,299,LEMD,ADOLFO SUÁREZ MADRID-BARAJAS AIRPORT - MADRID ...,01/02/2020 11:40,01/02/2020 12:21,SBRF,GUARARAPES - GILBERTO FREYRE - RECIFE - PE - B...,01/02/2020 20:15,01/02/2020 20:00,REALIZADO,SEGURANÇA/PAX/CARGA/ALARME,2020-02-01,Atraso 30-60,Antecipado,VRA_2020_02.csv,561


### 6. Investigar campos obrigatórios e datas previstas
O notebook de exploração de 2019 exige partida e chegada previstas. Seu ELT também rejeita ausência de companhia ICAO, aeroporto de origem ou destino. Usamos esses **cinco campos como referência provisória**, até acesso ao contrato e à modelagem. Não acrescentamos obrigatoriedade de número de voo ou horários reais por inferência.

Datas reais ausentes são analisadas por situação de voo; não são automaticamente rejeições (por exemplo, voos cancelados). Ausência e valor preenchido com formato inválido são problemas distintos. A estimativa abaixo usa apenas ausência e desconta a interseção com duplicatas excedentes para não subtrair registros duas vezes.

In [6]:
campos_obrigatorios = ["Partida Prevista", "Chegada Prevista", "Sigla ICAO Empresa Aérea",
                      "Sigla ICAO Aeroporto Origem", "Sigla ICAO Aeroporto Destino"]
sem_previstas = ausencias[["Partida Prevista", "Chegada Prevista"]].any(axis=1)
datas_operacao = ["Partida Prevista", "Partida Real", "Chegada Prevista", "Chegada Real"]
sem_quatro_datas = ausencias[datas_operacao].all(axis=1)
sem_obrigatorios = ausencias[campos_obrigatorios].any(axis=1)
resumo_obrigatorios = pd.DataFrame({
    "campo": campos_obrigatorios,
    "ausentes_na_bruta": ausencias[campos_obrigatorios].sum().values,
    "ausentes_sem_excedentes": ausencias.loc[~excedente, campos_obrigatorios].sum().values
})
display(resumo_obrigatorios)
print("Sem alguma data prevista:", int(sem_previstas.sum()))
print("Sem nenhuma das quatro datas:", int(sem_quatro_datas.sum()))
print("Sem algum obrigatório:", int(sem_obrigatorios.sum()))
display(df_2020.loc[sem_previstas, "Situação Voo"].value_counts(dropna=False)
        .rename_axis("situacao").reset_index(name="registros"))
display(ausencias[datas_operacao].groupby(df_2020["Situação Voo"]).sum())
display(df_2020.loc[sem_obrigatorios, ["arquivo_origem", "linha_origem", "Situação Voo"]
                   + campos_obrigatorios].head(10))
rejeitadas_apos_deduplicacao = int((sem_obrigatorios & ~excedente).sum())
intersecao = int((sem_obrigatorios & excedente).sum())
registros_analiticos_estimados = len(df_2020) - duplicatas_excedentes - rejeitadas_apos_deduplicacao
assert len(df_2020) == duplicatas_excedentes + rejeitadas_apos_deduplicacao + registros_analiticos_estimados
print("Sobreposição entre ausência obrigatória e excedentes:", intersecao)
print("Potenciais rejeições após deduplicar:", rejeitadas_apos_deduplicacao)
print("Contagem analítica estimada, apenas por essas regras:", registros_analiticos_estimados)

,campo,ausentes_na_bruta,ausentes_sem_excedentes
0,Partida Prevista,1512,1512
1,Chegada Prevista,1512,1512
2,Sigla ICAO Empresa Aérea,0,0
3,Sigla ICAO Aeroporto Origem,0,0
4,Sigla ICAO Aeroporto Destino,0,0


Sem alguma data prevista: 1512
Sem nenhuma das quatro datas: 5
Sem algum obrigatório: 1512


,situacao,registros
0,REALIZADO,1507
1,CANCELADO,5


,Partida Prevista,Partida Real,Chegada Prevista,Chegada Real
Situação Voo,,,,
CANCELADO,5,21510,5,21510
NÃO INFORMADO,0,358,0,358
REALIZADO,1507,0,1507,0


,arquivo_origem,linha_origem,Situação Voo,Partida Prevista,Chegada Prevista,Sigla ICAO Empresa Aérea,Sigla ICAO Aeroporto Origem,Sigla ICAO Aeroporto Destino
212909,VRA_2020_03.csv,39602,CANCELADO,,,JAT,SCEL,SBSV
212910,VRA_2020_03.csv,39603,CANCELADO,,,JAT,SBSV,SCEL
212911,VRA_2020_03.csv,39604,CANCELADO,,,JAT,SCEL,SBFI
212912,VRA_2020_03.csv,39605,CANCELADO,,,JAT,SBFI,SCEL
212913,VRA_2020_03.csv,39606,CANCELADO,,,JAT,SBSV,SCEL
242407,VRA_2020_04.csv,4577,REALIZADO,,,OWT,SBEG,SWPI
242414,VRA_2020_04.csv,4584,REALIZADO,,,OWT,SBVT,SBGR
242431,VRA_2020_04.csv,4601,REALIZADO,,,OWT,SBGR,SBVT
242454,VRA_2020_04.csv,4624,REALIZADO,,,OWT,SBEG,SWPI
242485,VRA_2020_04.csv,4655,REALIZADO,,,OWT,SBEG,SWPI


Sobreposição entre ausência obrigatória e excedentes: 0
Potenciais rejeições após deduplicar: 1512
Contagem analítica estimada, apenas por essas regras: 481909


### 7. Investigar diferenças de formato
A referência de 2019 usa `DD/MM/AAAA HH:MM` nas quatro datas operacionais e `AAAA-MM-DD` em `Referência`. Validamos separadamente a forma textual (regex) e a validade de calendário/horário (`to_datetime` em séries temporárias, com formato explícito). Um texto com espaços externos é divergente do padrão literal, mesmo que pudesse ser padronizado depois. Não há conversão de fuso.

Padrões de ICAO e de inteiros abaixo são **sondagens**, não novas regras de rejeição. Números de voo continuam textuais: zeros à esquerda não são erro. Também contamos espaços externos e variantes de caixa nos campos categóricos sem padronizá-los na base. A comparação com 2019 é documental: os CSVs daquele ano não estão neste checkout.

In [7]:
formatos_data = {c: (r"[0-9]{2}/[0-9]{2}/[0-9]{4} [0-9]{2}:[0-9]{2}", "%d/%m/%Y %H:%M")
                 for c in datas_operacao}
formatos_data["Referência"] = (r"[0-9]{4}-[0-9]{2}-[0-9]{2}", "%Y-%m-%d")
formatos_mensais = []
amostras_formatos = []
for coluna, (padrao, formato) in formatos_data.items():
    texto = df_2020[coluna]
    preenchido = ~ausencias[coluna]
    forma_valida = texto.str.fullmatch(padrao, na=False)
    # Valida valores distintos para evitar conversões repetidas, sem tocar a fonte.
    distintos = texto.loc[preenchido & forma_valida].unique()
    invalidos = set(distintos[pd.isna(pd.to_datetime(distintos, format=formato, errors="coerce", exact=True))])
    calendario_invalido = preenchido & forma_valida & texto.isin(invalidos)
    forma_divergente = preenchido & ~forma_valida
    for nome, indices in df_2020.groupby("arquivo_origem").groups.items():
        formatos_mensais.append({"arquivo": nome, "coluna": coluna,
            "ausentes": int(ausencias.loc[indices, coluna].sum()),
            "formato_divergente": int(forma_divergente.loc[indices].sum()),
            "calendario_horario_invalido": int(calendario_invalido.loc[indices].sum())})
    for motivo, mascara in [("formato_divergente", forma_divergente),
                             ("calendario_horario_invalido", calendario_invalido)]:
        for _, linha in df_2020.loc[mascara].head(5).iterrows():
            amostras_formatos.append({"arquivo": linha["arquivo_origem"], "linha": linha["linha_origem"],
                                     "coluna": coluna, "motivo": motivo, "valor_original": linha[coluna]})
resumo_formatos_mensais = pd.DataFrame(formatos_mensais)
resumo_formatos = resumo_formatos_mensais.groupby("coluna")[["ausentes", "formato_divergente", "calendario_horario_invalido"]].sum()
display(resumo_formatos)
display(resumo_formatos_mensais)
display(pd.DataFrame(amostras_formatos, columns=["arquivo", "linha", "coluna", "motivo", "valor_original"]))

,ausentes,formato_divergente,calendario_horario_invalido
coluna,,,
Chegada Prevista,1512,0,0
Chegada Real,21868,0,0
Partida Prevista,1512,0,0
Partida Real,21868,0,0
Referência,0,0,0


,arquivo,coluna,ausentes,formato_divergente,calendario_horario_invalido
0,VRA_2020_01.csv,Partida Prevista,0,0,0
1,VRA_2020_02.csv,Partida Prevista,0,0,0
2,VRA_2020_03.csv,Partida Prevista,5,0,0
3,VRA_2020_04.csv,Partida Prevista,66,0,0
4,VRA_2020_05.csv,Partida Prevista,70,0,0
...,...,...,...,...,...
55,VRA_2020_08.csv,Referência,0,0,0
56,VRA_2020_09.csv,Referência,0,0,0
57,VRA_2020_10.csv,Referência,0,0,0
58,VRA_2020_11.csv,Referência,0,0,0


,arquivo,linha,coluna,motivo,valor_original


In [8]:
padroes_campos = {
    "Sigla ICAO Empresa Aérea": r"[A-Z]{3}",
    "Sigla ICAO Aeroporto Origem": r"[A-Z]{4}",
    "Sigla ICAO Aeroporto Destino": r"[A-Z]{4}",
    "Número Voo": r"[0-9]+", "Número de Assentos": r"[0-9]+", "Código DI": r"[0-9]+"
}
sondagens = []
for coluna, padrao in padroes_campos.items():
    divergente = ~ausencias[coluna] & ~df_2020[coluna].str.fullmatch(padrao, na=False)
    sondagens.append({"coluna": coluna, "padrao_exploratorio": padrao,
                      "preenchidos_divergentes": int(divergente.sum()),
                      "exemplos_originais": df_2020.loc[divergente, coluna].drop_duplicates().head(5).tolist()})
display(pd.DataFrame(sondagens))
print("Números de voo com zeros à esquerda:", int(df_2020["Número Voo"].str.match(r"0[0-9]+").sum()))
resumo_espacos = pd.DataFrame({
    "coluna": colunas_originais,
    "com_espacos_externos": [int(df_2020[c].ne(df_2020[c].str.strip()).sum()) for c in colunas_originais]
})
display(resumo_espacos)
for coluna in ["Código Tipo Linha", "Situação Voo", "Situação Partida", "Situação Chegada"]:
    print("Valores originais de", coluna)
    display(df_2020[coluna].value_counts(dropna=False).rename_axis("valor_original").reset_index(name="registros"))
    variantes = pd.DataFrame({"original": df_2020[coluna].unique()})
    variantes["comparacao_temporaria"] = variantes["original"].str.strip().str.upper()
    display(variantes.loc[variantes.duplicated("comparacao_temporaria", keep=False)])

,coluna,padrao_exploratorio,preenchidos_divergentes,exemplos_originais
0,Sigla ICAO Empresa Aérea,[A-Z]{3},0,[]
1,Sigla ICAO Aeroporto Origem,[A-Z]{4},0,[]
2,Sigla ICAO Aeroporto Destino,[A-Z]{4},0,[]
3,Número Voo,[0-9]+,561,"[Z2565, Z4877, Z2832, Z4307, Z9047]"
4,Número de Assentos,[0-9]+,0,[]
5,Código DI,[0-9]+,25,"[B, A, E]"


Números de voo com zeros à esquerda: 28835


,coluna,com_espacos_externos
0,Sigla ICAO Empresa Aérea,0
1,Empresa Aérea,0
2,Número Voo,0
3,Código DI,24
4,Código Tipo Linha,0
5,Modelo Equipamento,0
6,Número de Assentos,0
7,Sigla ICAO Aeroporto Origem,0
8,Descrição Aeroporto Origem,0
9,Partida Prevista,0


Valores originais de Código Tipo Linha


,valor_original,registros
0,N,401709
1,I,38490
2,X,17642
3,C,14913
4,G,10348
5,H,283
6,L,129
7,R,24
8,E,5
9,,2


,original,comparacao_temporaria


... (5 outputs left out)

,original,comparacao_temporaria


Valores originais de Situação Chegada


,valor_original,registros
0,Antecipado,267823
1,Pontual,164305
2,,23375
3,Atraso 30-60,15261
4,Atraso 60-120,7036
5,Atraso 120-240,3802
6,Atraso >240,1943


,original,comparacao_temporaria


### 8. Resumo para o relatório e verificação da preservação
A contagem analítica é uma simulação aritmética das regras provisórias, não uma base tratada nem garantia de conformidade com o futuro contrato. Duplicatas e ausências permanecem integralmente em `df_2020`. A assinatura do DataFrame e os hashes dos CSVs devem ser iguais aos capturados na leitura.

In [9]:
diagnostico = pd.DataFrame([
    ("ano", 2020), ("arquivos", len(arquivos)), ("registros_brutos", len(df_2020)),
    ("colunas_originais", len(colunas_originais)),
    ("grupos_repetidos", grupos_repetidos), ("duplicatas_envolvidos", registros_envolvidos),
    ("duplicatas_excedentes", duplicatas_excedentes),
    ("sem_alguma_data_prevista", int(sem_previstas.sum())),
    ("sem_quatro_datas", int(sem_quatro_datas.sum())),
    ("sem_algum_obrigatorio_provisorio", int(sem_obrigatorios.sum())),
    ("ausencias_obrigatorias_em_excedentes", intersecao),
    ("rejeitadas_apos_deduplicacao_estimativa", rejeitadas_apos_deduplicacao),
    ("registros_analiticos_estimados", registros_analiticos_estimados),
    ("datas_com_formato_divergente", int(resumo_formatos["formato_divergente"].sum())),
    ("datas_com_calendario_horario_invalido", int(resumo_formatos["calendario_horario_invalido"].sum()))
], columns=["indicador", "valor"])
display(diagnostico)
assert len(df_2020) == resumo_estrutura["registros"].sum()
assert not df_2020[colunas_originais].isna().any().any()
assert assinatura_dataframe(df_2020) == assinatura_inicial, "O DataFrame original foi alterado."
assert {p.name: sha256_arquivo(p) for p in arquivos} == hashes_iniciais, "Os CSVs foram alterados."
print("Exploração concluída: DataFrame e CSVs originais preservados; nenhum ETL executado.")

,indicador,valor
0,ano,2020
1,arquivos,12
2,registros_brutos,483545
3,colunas_originais,20
4,grupos_repetidos,124
5,duplicatas_envolvidos,248
6,duplicatas_excedentes,124
7,sem_alguma_data_prevista,1512
8,sem_quatro_datas,5
9,sem_algum_obrigatorio_provisorio,1512


Exploração concluída: DataFrame e CSVs originais preservados; nenhum ETL executado.


## Decisões e próxima etapa


**Evidências desta execução:** 483.545 registros em 12 arquivos com 20 colunas originais; 124 grupos duplicados (248 registros envolvidos e 124 excedentes); 1.512 registros sem ambas as datas previstas, dos quais 5 sem nenhuma das quatro datas. Não há sobreposição dessas ausências com excedentes. A estimativa provisória é 481.909 registros analíticos. Datas preenchidas seguem os formatos de referência e passaram na validação de calendário/horário. Esses resultados descrevem os arquivos identificados pelos hashes acima.

- **Preservação e auditoria:** texto integral, sem limpeza, com arquivo e posição lógica de origem. Datas e números não são convertidos na base original.
- **Comparabilidade com 2019:** mantidas as análises de volume, vazios, duplicatas exatas e datas previstas, ampliadas com estrutura mensal, outros obrigatórios e formatos. Totais de 2019 não são metas para 2020.
- **Ausências:** campos opcionais e datas reais não recebem rejeição automática. As distribuições por situação ajudam a interpretar o preenchimento; causas precisam de confirmação e não são inferidas apenas das contagens.
- **Duplicatas:** excedentes anuais são contados nos campos originais, antes de qualquer padronização. A eventual remoção caberá somente à camada tratada, mediante contrato.
- **Formatos:** vazios, divergências textuais e datas impossíveis são contabilizados separadamente. Sondagens de ICAO/inteiros não definem obrigatoriedade ou validade de domínio.
- **Pendência:** conferir contrato, dicionário e esquema estrela antes de fixar tipos finais, chaves da fato/dimensões, rejeições e regras de negócio. A estimativa analítica considera somente os cinco obrigatórios usados no ELT de 2019.

O próximo trabalho será revisar estas evidências com a documentação do grupo. O pipeline ETL, a exportação de dados tratados e a carga em banco ficam para uma etapa posterior.